# **Stage 3 - Feature Engineering**

This section creates the final modeling features for the FOODS demand forecasting task. Based on the EDA findings, the feature set includes calendar features, lag features, rolling statistics, price features, event features, SNAP indicators, and static product-store identifiers.

In [1]:
# feature engineering
import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("outputs")
df = pd.read_parquet(OUT / "foods_clean.parquet")

# CRITICAL: sort by id then date — all lag/rolling features depend on this order
df = df.sort_values(["id", "date"]).reset_index(drop=True)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (21798313, 22)
Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'd_num', 'date', 'wm_yr_wk', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price']


### 3.1 Calendar features

Calendar features are added to capture time-based demand patterns. These include `dayofweek`, `is_weekend`, `month`, and `dayofyear`. These variables are useful because the EDA showed clear weekly and monthly seasonality in food demand.

In [2]:
# Add calendar features
from calendar_features import add_calendar_features

df = add_calendar_features(df)

cal_feats = ["dayofweek", "is_weekend", "month", "dayofyear"]
print("New calendar features:", cal_feats)
df[["date"] + cal_feats].head(8)

New calendar features: ['dayofweek', 'is_weekend', 'month', 'dayofyear']


,date,dayofweek,is_weekend,month,dayofyear
0,2011-01-29,5,1,1,29
1,2011-01-30,6,1,1,30
2,2011-01-31,0,0,1,31
3,2011-02-01,1,0,2,32
4,2011-02-02,2,0,2,33
5,2011-02-03,3,0,2,34
6,2011-02-04,4,0,2,35
7,2011-02-05,5,1,2,36


### 3.2 Lag Features

Lag features are created using sales from 7, 14, and 28 days before the current date. These features allow the model to learn from recent historical demand patterns. The 7-day lag captures weekly repetition, while the 14-day and 28-day lags help capture longer short-term trends.

In [3]:
# Add lag features
from lag_features import add_lag_features

df = add_lag_features(df, lags=(7, 14, 28))

lag_feats = ["lag_7", "lag_14", "lag_28"]
print("Lag features added:", lag_feats)
# look at one series to see lags shifting in
one = df[df["id"] == df["id"].iloc[0]][["date", "sales"] + lag_feats].head(35)
one

Lag features added: ['lag_7', 'lag_14', 'lag_28']


,date,sales,lag_7,lag_14,lag_28
0,2011-01-29,3,NaN,NaN,NaN
1,2011-01-30,0,NaN,NaN,NaN
2,2011-01-31,0,NaN,NaN,NaN
3,2011-02-01,1,NaN,NaN,NaN
4,2011-02-02,4,NaN,NaN,NaN
5,2011-02-03,2,NaN,NaN,NaN
6,2011-02-04,0,NaN,NaN,NaN
7,2011-02-05,2,3.0,NaN,NaN
8,2011-02-06,0,0.0,NaN,NaN
9,2011-02-07,0,0.0,NaN,NaN


### 3.3 Rolling Features

Rolling mean and rolling standard deviation features are created using 7-day and 28-day windows. Rolling means help capture recent average demand, while rolling standard deviations measure recent demand volatility. These features are important because food demand changes over time and may fluctuate differently across products and stores.


In [4]:
# Add rolling features
from rolling_features import add_rolling_features

df = add_rolling_features(df, windows=(7, 28))

roll_feats = ["roll_mean_7", "roll_std_7", "roll_mean_28", "roll_std_28"]
print("Rolling features added:", roll_feats)
one = df[df["id"] == df["id"].iloc[0]][["date", "sales"] + roll_feats].head(35)
one

Rolling features added: ['roll_mean_7', 'roll_std_7', 'roll_mean_28', 'roll_std_28']


,date,sales,roll_mean_7,roll_std_7,roll_mean_28,roll_std_28
0,2011-01-29,3,NaN,NaN,NaN,NaN
1,2011-01-30,0,NaN,NaN,NaN,NaN
2,2011-01-31,0,NaN,NaN,NaN,NaN
3,2011-02-01,1,NaN,NaN,NaN,NaN
4,2011-02-02,4,NaN,NaN,NaN,NaN
5,2011-02-03,2,NaN,NaN,NaN,NaN
6,2011-02-04,0,NaN,NaN,NaN,NaN
7,2011-02-05,2,1.428571,1.618347,NaN,NaN
8,2011-02-06,0,1.285714,1.496027,NaN,NaN
9,2011-02-07,0,1.285714,1.496027,NaN,NaN


### 3.4 Price Features

Price-related features are added using `sell_price` and `price_vs_median`. The `price_vs_median` feature compares the current price with the product’s typical price level. This is useful because the EDA showed a negative relationship between price and demand, suggesting that price changes may help explain sales variation.

In [5]:
# Add price features
from price_feature import add_price_features

df = add_price_features(df)
df[["id", "date", "sell_price", "price_vs_median"]].head(10)

,id,date,sell_price,price_vs_median
0,FOODS_1_001_CA_1_evaluation,2011-01-29,2.0,NaN
1,FOODS_1_001_CA_1_evaluation,2011-01-30,2.0,NaN
2,FOODS_1_001_CA_1_evaluation,2011-01-31,2.0,NaN
3,FOODS_1_001_CA_1_evaluation,2011-02-01,2.0,NaN
4,FOODS_1_001_CA_1_evaluation,2011-02-02,2.0,NaN
5,FOODS_1_001_CA_1_evaluation,2011-02-03,2.0,NaN
6,FOODS_1_001_CA_1_evaluation,2011-02-04,2.0,NaN
7,FOODS_1_001_CA_1_evaluation,2011-02-05,2.0,NaN
8,FOODS_1_001_CA_1_evaluation,2011-02-06,2.0,NaN
9,FOODS_1_001_CA_1_evaluation,2011-02-07,2.0,NaN


### 3.5 Event Features

### Event Column Inspection

Most dates do not have events, with about 91.9% missing values in the primary event columns and 99.8% missing values in the secondary event columns. This means that missing values mostly represent normal non-event days rather than data quality problems.

In [6]:
# Inspect event columns
for c in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    n = df[c].nunique()
    miss = df[c].isna().mean() * 100
    print(f"{c}: {n} unique values, {miss:.1f}% missing")

print("\nEvent types (type_1):")
print(df["event_type_1"].value_counts(dropna=False))

event_name_1: 30 unique values, 91.9% missing
event_type_1: 4 unique values, 91.9% missing
event_name_2: 4 unique values, 99.8% missing
event_type_2: 2 unique values, 99.8% missing

Event types (type_1):
event_type_1
NaN          20026431
Religious      604112
National       577235
Cultural       416444
Sporting       174091
Name: count, dtype: int64


### Event Feature Encoding

Only the primary event columns are kept because secondary event columns are almost all missing. Missing event values are encoded as `none`, allowing the model to distinguish normal days from event days.

In [7]:
# Apply event encoding (primary event only)
from event_feature import add_event_features

df = add_event_features(df)

print("Kept event columns:", [c for c in df.columns if c.startswith("event")])
print(f"event_type_1 categories: {df['event_type_1'].cat.categories.tolist()}")
print(f"event_name_1 categories: {df['event_name_1'].nunique()}")

Kept event columns: ['event_name_1', 'event_type_1']
event_type_1 categories: ['Cultural', 'National', 'Religious', 'Sporting', 'none']
event_name_1 categories: 31


### 3.6 Snap Features

About 32.9% of observations occur on SNAP days. Since the EDA showed that SNAP days are associated with higher demand, this feature may help the model capture external demand shifts.

In [8]:
# rebuild single snap column from per-state columns
if {"snap_CA", "snap_TX", "snap_WI"}.issubset(df.columns):
    df["snap"] = np.select(
        [df["state_id"] == "CA", df["state_id"] == "TX", df["state_id"] == "WI"],
        [df["snap_CA"], df["snap_TX"], df["snap_WI"]],
        default=0,
    ).astype("int8")
    df = df.drop(columns=["snap_CA", "snap_TX", "snap_WI"])
    print("snap column rebuilt.")
elif "snap" in df.columns:
    print("snap already exists.")
else:
    print("WARNING: no snap columns found at all — check the parquet.")

print([c for c in df.columns if "snap" in c.lower()])

snap column rebuilt.
['snap']


In [9]:
# === Cell 52: Confirm SNAP is ready (already clean 0/1) ===
print("snap dtype:", df["snap"].dtype)
print(df["snap"].value_counts())
print(f"SNAP-day share: {df['snap'].mean()*100:.1f}%")

snap dtype: int8
snap
0    14622026
1     7176287
Name: count, dtype: int64
SNAP-day share: 32.9%


### 3.7 Static Identifier Features

Static identifier features include `item_id`, `dept_id`, `store_id`, and `state_id`. These variables help the model learn differences across products, departments, stores, and states. The `cat_id` column is excluded because it is constant after filtering the dataset to the FOODS category.

In [10]:
# Confirm static identifier features
static_feats = ["item_id", "dept_id", "store_id", "state_id"]
for c in static_feats:
    print(f"{c}: dtype={df[c].dtype}, {df[c].nunique()} categories")
# cat_id is constant (all FOODS) — useless as feature, note it
print(f"\ncat_id unique: {df['cat_id'].nunique()} (constant, will exclude from features)")

item_id: dtype=category, 1437 categories
dept_id: dtype=category, 3 categories
store_id: dtype=category, 10 categories
state_id: dtype=category, 3 categories

cat_id unique: 1 (constant, will exclude from features)


### 3.8 Final Feature Set and Check

The final feature set contains 20 features grouped into calendar, lag, rolling, price, event, SNAP, and static identifier features. This feature design reflects the main EDA findings: demand is affected by time patterns, recent sales history, price changes, events, SNAP activity, and product-store differences.

In [11]:
# Define the final feature set
target = "sales"

# Feature groups (only evidence-backed features we built)
calendar_feats = ["dayofweek", "is_weekend", "month", "dayofyear"]
lag_feats      = ["lag_7", "lag_14", "lag_28"]
roll_feats     = ["roll_mean_7", "roll_std_7", "roll_mean_28", "roll_std_28"]
price_feats    = ["sell_price", "price_vs_median"]
event_feats    = ["event_name_1", "event_type_1"]
snap_feats     = ["snap"]
static_feats   = ["item_id", "dept_id", "store_id", "state_id"]

feature_cols = (calendar_feats + lag_feats + roll_feats +
                price_feats + event_feats + snap_feats + static_feats)

print(f"Total features: {len(feature_cols)}")
for grp, feats in [("calendar", calendar_feats), ("lag", lag_feats),
                   ("rolling", roll_feats), ("price", price_feats),
                   ("event", event_feats), ("snap", snap_feats),
                   ("static", static_feats)]:
    print(f"  {grp}: {feats}")

Total features: 20
  calendar: ['dayofweek', 'is_weekend', 'month', 'dayofyear']
  lag: ['lag_7', 'lag_14', 'lag_28']
  rolling: ['roll_mean_7', 'roll_std_7', 'roll_mean_28', 'roll_std_28']
  price: ['sell_price', 'price_vs_median']
  event: ['event_name_1', 'event_type_1']
  snap: ['snap']
  static: ['item_id', 'dept_id', 'store_id', 'state_id']




A final integrity check confirms that all selected feature columns are present.

In [12]:
# === Cell 55: Final integrity check ===
# all feature columns present?
missing = [c for c in feature_cols if c not in df.columns]
print("Missing feature columns:", missing if missing else "none ✓")

# NaN summary (lag/rolling have expected early-series NaN; others should be ~0)
print("\nNaN ratio per feature:")
nan_summary = df[feature_cols].isna().mean().sort_values(ascending=False)
print(nan_summary[nan_summary > 0])
print("\nFeatures with zero NaN:", (nan_summary == 0).sum(), "/", len(feature_cols))

Missing feature columns: none ✓

NaN ratio per feature:
roll_mean_28       0.018458
roll_std_28        0.018458
lag_28             0.018458
lag_14             0.009229
price_vs_median    0.009229
roll_mean_7        0.004615
roll_std_7         0.004615
lag_7              0.004615
dtype: float64

Features with zero NaN: 12 / 20


### 3.9 Save Final Feature Matrix

The final feature matrix is saved as `foods_features.parquet`. It contains `id`, `date`, `d_num`, the target variable `sales`, and all selected modeling features. The final dataset has 21,798,313 rows and 24 columns, making it ready for train-test splitting and model development.

In [13]:
# Save final feature matrix
# keep: id/date for indexing & splitting, target, and all features
keep_cols = ["id", "date", "d_num", target] + feature_cols
# dedupe while preserving order
keep_cols = list(dict.fromkeys(keep_cols))

final = df[keep_cols].copy()
final.to_parquet(OUT / "foods_features.parquet", index=False)

print(f"Saved foods_features.parquet")
print(f"Shape: {final.shape}")
print(f"Memory: {final.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Columns: {final.columns.tolist()}")

Saved foods_features.parquet
Shape: (21798313, 24)
Memory: 1.42 GB
Columns: ['id', 'date', 'd_num', 'sales', 'dayofweek', 'is_weekend', 'month', 'dayofyear', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_std_7', 'roll_mean_28', 'roll_std_28', 'sell_price', 'price_vs_median', 'event_name_1', 'event_type_1', 'snap', 'item_id', 'dept_id', 'store_id', 'state_id']
